# 🌾 AgriData Sénégal — Notebook de présentation de l’interface

## Projet Architecture Big Data — Agriculture de précision

Ce notebook présente la logique de l’interface **AgriData Sénégal**, un tableau de bord destiné à un responsable agricole.

L’objectif n’est pas seulement d’afficher des données, mais d’aider le patron à répondre rapidement à ces questions :

- Quels champs vont bien ?
- Quels champs sont en danger ?
- Quelle est la cause du problème ?
- Quelle action faut-il prévoir ?
- Quel champ doit être traité en priorité ?

## 1. Rôle de l’interface dans l’architecture

L’interface correspond à la partie visible du projet.

Dans le pipeline Big Data, les données sont produites par :

- les capteurs de sol : humidité, température du sol, pH ;
- les données météo : température de l’air, pluie ;
- les données de production : surface, rendement, production estimée.

Ces données sont ensuite utilisées par le dashboard pour fournir une lecture métier claire.

### Pipeline simplifié

```text
Capteurs sol + Météo + Production
        ↓
Redpanda / Kafka
        ↓
Traitement Spark
        ↓
Données prêtes à exploiter
        ↓
Dashboard AgriData Sénégal
```

## 2. Importation des librairies

On utilise principalement **pandas** pour préparer les données et calculer les indicateurs utilisés dans le dashboard.

In [ ]:
import pandas as pd

## 3. Données utilisées par l’interface

Dans cette version de démonstration, les données sont construites à partir des informations prévues dans les scripts du projet :

- données de sol ;
- météo agricole ;
- production agricole ;
- nom du champ ;
- région ;
- culture.

Chaque ligne représente un champ suivi par **AgriData Sénégal**.

In [ ]:
data = [
    {
        "champ": "Champ Louga Nord",
        "region": "Louga",
        "culture": "Mil",
        "humidite_sol": 15,
        "temperature_sol": 33,
        "temperature_air": 36,
        "pluie_mm": 0.0,
        "ph_sol": 6.9,
        "rendement": 0.70,
        "production_t": 6300,
        "surface_ha": 9000
    },
    {
        "champ": "Rizière Ziguinchor Sud",
        "region": "Ziguinchor",
        "culture": "Riz",
        "humidite_sol": 75,
        "temperature_sol": 27,
        "temperature_air": 29,
        "pluie_mm": 5.5,
        "ph_sol": 4.5,
        "rendement": 2.80,
        "production_t": 42000,
        "surface_ha": 15000
    },
    {
        "champ": "Zone Matam Est",
        "region": "Matam",
        "culture": "Sorgho",
        "humidite_sol": 42,
        "temperature_sol": 34,
        "temperature_air": 37,
        "pluie_mm": 0.5,
        "ph_sol": 7.0,
        "rendement": 0.80,
        "production_t": 6400,
        "surface_ha": 8000
    },
    {
        "champ": "Champ Kaolack Centre",
        "region": "Kaolack",
        "culture": "Arachide",
        "humidite_sol": 55,
        "temperature_sol": 28,
        "temperature_air": 32,
        "pluie_mm": 2.5,
        "ph_sol": 6.5,
        "rendement": 1.20,
        "production_t": 14400,
        "surface_ha": 12000
    },
    {
        "champ": "Rizière Saint-Louis",
        "region": "Saint-Louis",
        "culture": "Riz",
        "humidite_sol": 60,
        "temperature_sol": 26,
        "temperature_air": 31,
        "pluie_mm": 1.2,
        "ph_sol": 7.2,
        "rendement": 2.50,
        "production_t": 45000,
        "surface_ha": 18000
    },
    {
        "champ": "Périmètre Dakar Maraîchage",
        "region": "Dakar",
        "culture": "Maraîchage",
        "humidite_sol": 62,
        "temperature_sol": 27,
        "temperature_air": 30,
        "pluie_mm": 3.0,
        "ph_sol": 7.1,
        "rendement": 10.00,
        "production_t": 30000,
        "surface_ha": 3000
    }
]

df = pd.DataFrame(data)
df

## 4. Règles métier utilisées pour détecter les risques

L’interface doit transformer les données en informations utiles.

On définit donc des règles simples :

| Indicateur | Condition | Interprétation |
|---|---:|---|
| Humidité du sol | `< 20 %` | risque de sécheresse |
| Pluie + chaleur | pluie `< 1 mm` et température air `> 35°C` | risque climatique |
| pH du sol | `< 5.0` ou `> 8.5` | sol déséquilibré |
| Température de l’air | `> 35°C` | stress thermique |
| Rendement | `< 0.8 t/ha` | rendement faible |

Ces règles permettent de calculer un **score de risque** pour chaque champ.

In [ ]:
df["risque_secheresse"] = (df["humidite_sol"] < 20) | ((df["pluie_mm"] < 1) & (df["temperature_air"] > 35))
df["probleme_ph"] = (df["ph_sol"] < 5.0) | (df["ph_sol"] > 8.5)
df["stress_chaleur"] = df["temperature_air"] > 35
df["rendement_faible"] = df["rendement"] < 0.8

df["score_risque"] = (
    df["risque_secheresse"].astype(int)
    + df["probleme_ph"].astype(int)
    + df["stress_chaleur"].astype(int)
    + df["rendement_faible"].astype(int)
)

df[["champ", "risque_secheresse", "probleme_ph", "stress_chaleur", "rendement_faible", "score_risque"]]

## 5. Niveau d’état du champ

À partir du score de risque, on classe chaque champ :

- **BON** : aucun problème détecté ;
- **À SURVEILLER** : un ou deux signaux faibles ;
- **CRITIQUE** : plusieurs problèmes simultanés.

Cette classification est importante pour le patron, car elle permet de prioriser les interventions.

In [ ]:
def niveau(score):
    if score >= 3:
        return "CRITIQUE"
    elif score >= 1:
        return "À SURVEILLER"
    return "BON"

df["etat"] = df["score_risque"].apply(niveau)

df[["champ", "region", "culture", "score_risque", "etat"]]

## 6. Diagnostic automatique

Le dashboard ne doit pas seulement dire qu’un champ est en danger.

Il doit aussi expliquer **pourquoi**.

On crée donc une colonne `probleme_principal` qui résume les problèmes détectés pour chaque champ.

In [ ]:
def probleme(row):
    problemes = []

    if row["risque_secheresse"]:
        problemes.append("Sol trop sec + manque de pluie")
    if row["stress_chaleur"]:
        problemes.append("Température élevée")
    if row["probleme_ph"]:
        problemes.append("pH du sol déséquilibré")
    if row["rendement_faible"]:
        problemes.append("Rendement faible")

    return " + ".join(problemes) if problemes else "Aucun problème majeur"

df["probleme_principal"] = df.apply(probleme, axis=1)

df[["champ", "etat", "probleme_principal"]]

## 7. Actions recommandées

Pour rendre l’interface utile au patron, on ajoute une recommandation opérationnelle.

Le but est de dire clairement ce qu’il faut faire :

- irriguer ;
- corriger le pH ;
- surveiller la météo ;
- apporter des nutriments ;
- maintenir les pratiques actuelles.

In [ ]:
def action(row):
    actions = []

    if row["risque_secheresse"]:
        actions.append("Irriguer sous 24h")
    if row["stress_chaleur"]:
        actions.append("Paillage + suivi météo")
    if row["probleme_ph"]:
        actions.append("Corriger le pH du sol")
    if row["rendement_faible"]:
        actions.append("Apport nutriments")

    return " ; ".join(actions) if actions else "Maintenir pratiques"

df["action_recommandee"] = df.apply(action, axis=1)

df[["champ", "etat", "probleme_principal", "action_recommandee"]]

## 8. Tableau de pilotage principal

Ce tableau correspond à la vue que le patron doit voir en priorité.

Il affiche :

- le nom du champ ;
- la région ;
- la culture ;
- l’état du champ ;
- le problème détecté ;
- l’action recommandée ;
- le rendement ;
- la production estimée.

In [ ]:
tableau_pilotage = df[
    [
        "champ",
        "region",
        "culture",
        "etat",
        "probleme_principal",
        "action_recommandee",
        "rendement",
        "production_t"
    ]
].sort_values(by="etat")

tableau_pilotage

## 9. Indicateurs clés du dashboard

Les cartes KPI du dashboard donnent une vue rapide de la situation globale.

Ces indicateurs sont destinés à être lus en quelques secondes.

In [ ]:
nb_champs = len(df)
nb_critiques = len(df[df["etat"] == "CRITIQUE"])
production_totale = df["production_t"].sum()
rendement_moyen = df["rendement"].mean()

print("Champs suivis :", nb_champs)
print("Champs critiques :", nb_critiques)
print("Production totale :", production_totale, "tonnes")
print("Rendement moyen :", round(rendement_moyen, 2), "t/ha")

## 10. Champ prioritaire

L’interface doit aider à décider où intervenir en premier.

On trie donc les champs selon le score de risque.

In [ ]:
df_priorite = df.sort_values(by="score_risque", ascending=False)
champ_prioritaire = df_priorite.iloc[0]

print("Champ prioritaire :", champ_prioritaire["champ"])
print("Région :", champ_prioritaire["region"])
print("Culture :", champ_prioritaire["culture"])
print("État :", champ_prioritaire["etat"])
print("Problème :", champ_prioritaire["probleme_principal"])
print("Action recommandée :", champ_prioritaire["action_recommandee"])

## 11. Répartition des champs par état

Cette analyse permet de voir rapidement combien de champs sont :

- critiques ;
- à surveiller ;
- bons.

In [ ]:
repartition_etat = df["etat"].value_counts().reset_index()
repartition_etat.columns = ["etat", "nombre"]
repartition_etat

## 12. Production par culture

Cette vue permet d’identifier les cultures qui contribuent le plus à la production totale.

In [ ]:
production_culture = df.groupby("culture", as_index=False)["production_t"].sum()
production_culture = production_culture.sort_values(by="production_t", ascending=False)
production_culture

## 13. Détail d’un champ

Dans l’application Streamlit, cette partie correspond au bloc **Détail d’un champ**.

Le patron peut sélectionner un champ et obtenir :

- le nom du champ ;
- son état ;
- ses indicateurs clés ;
- ses problèmes ;
- ses actions recommandées ;
- une prévision court terme.

In [ ]:
champ_selectionne = "Champ Louga Nord"
champ = df[df["champ"] == champ_selectionne].iloc[0]

print("Nom du champ :", champ["champ"])
print("Région :", champ["region"])
print("Culture :", champ["culture"])
print("État :", champ["etat"])
print("Humidité du sol :", champ["humidite_sol"], "%")
print("Température de l'air :", champ["temperature_air"], "°C")
print("Pluie :", champ["pluie_mm"], "mm")
print("pH du sol :", champ["ph_sol"])
print("Rendement :", champ["rendement"], "t/ha")
print("Production estimée :", champ["production_t"], "tonnes")
print("Problème principal :", champ["probleme_principal"])
print("Action recommandée :", champ["action_recommandee"])

## 14. Prévision court terme

La prévision n’est pas une IA avancée ici.

C’est une prévision métier simple basée sur les risques détectés.

Elle permet d’aider le patron à anticiper les conséquences si aucune action n’est faite.

In [ ]:
def prevision_court_terme(row):
    if row["risque_secheresse"]:
        return "Sans intervention, risque de baisse du rendement dans les prochains jours."
    elif row["probleme_ph"]:
        return "Le pH peut limiter l’absorption des nutriments. Une correction est recommandée."
    elif row["stress_chaleur"]:
        return "La chaleur peut augmenter l’évaporation et fragiliser la culture."
    else:
        return "Conditions stables à court terme."

df["prevision_court_terme"] = df.apply(prevision_court_terme, axis=1)

df[["champ", "etat", "prevision_court_terme"]]

## 15. Conclusion

L’interface **AgriData Sénégal** a été conçue comme un outil d’aide à la décision pour un responsable agricole.

Elle permet de :

- visualiser l’état des champs ;
- identifier les champs critiques ;
- comprendre les causes des problèmes ;
- recommander des actions concrètes ;
- prioriser les interventions ;
- suivre la production et le rendement.

Le dashboard final est développé avec **Streamlit** et reprend cette logique sous forme d’une interface esthétique, sombre, verte et orientée métier.

## 16. Lancement du dashboard Streamlit

Pour lancer l’application depuis le terminal VS Code :

```bash
python3 -m streamlit run dashboard.py
```

Le dashboard s’ouvre ensuite sur une page locale de type :

```text
http://localhost:8501
```